In [42]:
users_interests = [
    ["Hadoop", "Big Data", "HBase", "Java", "Spark", "Storm", "Cassandra"],
    ["NoSQL", "MongoDB", "Cassandra", "HBase", "Postgres"],
    ["Python", "scikit-learn", "scipy", "numpy", "statsmodels", "pandas"],
    ["R", "Python", "statistics", "regression", "probability"],
    ["machine learning", "regression", "decision trees", "libsvm"],
    ["Python", "R", "Java", "C++", "Haskell", "programming languages"],
    ["statistics", "probability", "mathematics", "theory"],
    ["machine learning", "scikit-learn", "Mahout", "neural networks"],
    ["neural networks", "deep learning", "Big Data", "artificial intelligence"],
    ["Hadoop", "Java", "MapReduce", "Big Data"],
    ["statistics", "R", "statsmodels"],
    ["C++", "deep learning", "artificial intelligence", "probability"],
    ["pandas", "R", "Python"],
    ["databases", "HBase", "Postgres", "MySQL", "MongoDB"],
    ["libsvm", "regression", "support vector machines"]
]

In [43]:
from collections import Counter


popular_interests = Counter(interest 
                            for users_interest in users_interests for interest in users_interest)
popular_interests.most_common()[0]

('Python', 4)

In [44]:
from typing import List, Tuple

def most_popular_new_interest(popular_interests: Counter, users_interest:List[str], max_results: int = 5) ->List[Tuple[str, int]]:
    'sugeruje 5 najpolrajniejszych pozycji ktorych nie ma w swojej liscie zainteresowan'
    suggestions = [interest for interest in popular_interests.most_common() if interest[0] not in users_interest]
    return suggestions[:max_results]

most_popular_new_interest(popular_interests, ["NoSQL", "MongoDB", "Cassandra", "HBase", "Postgres"])

[('Python', 4), ('R', 4), ('Big Data', 3), ('Java', 3), ('statistics', 3)]

In [45]:
#user_based collaborative filtering

unique_interests = sorted({interest for user_interest in users_interests for interest in user_interest}) # sortownie alfabeycznie
unique_interests[:5] 

['Big Data', 'C++', 'Cassandra', 'HBase', 'Hadoop']

In [46]:
import numpy as np

def cosine_sim(a, b):
    return (np.dot(a,b)) / (np.linalg.norm(a) * np.linalg.norm(b))

def make_user_interest_vector(users_interest:List[str]) -> List[int]:
    'wektor na podstawie unique_interests warotsci 1 jezeli zainteresowania sie pokrywaja else 0'
    return [1 if interest in users_interest else 0 for interest in unique_interests]

users_interest_vectors = [make_user_interest_vector(interest) for interest in users_interests]

user_similarities = [[cosine_sim(i,j) for j in users_interest_vectors ]for i in users_interest_vectors]

# Users 0 and 9 share interests in Hadoop, Java, and Big Data
assert 0.56 < user_similarities[0][9] < 0.58, "several shared interests"

# Users 0 and 8 share only one interest: Big Data
assert 0.18 < user_similarities[0][8] < 0.20, "only one shared interest"

In [47]:
def most_similar_user_to(user_id:int) -> List[Tuple[int, float]]:
    "wektory i id uzydkownikow ktorzy maja najwiecej wspolnych zainteresowan"
    return sorted([(id, val) for id, val in enumerate(user_similarities[user_id]) if id != user_id and val >0], key=lambda x: x[-1], reverse=True)
most_similar_user_to(0)

[(9, np.float64(0.5669467095138409)),
 (1, np.float64(0.33806170189140655)),
 (8, np.float64(0.1889822365046136)),
 (13, np.float64(0.16903085094570328)),
 (5, np.float64(0.1543033499620919))]

In [48]:
from typing import Dict
from collections import defaultdict
def user_based_suggestions(user_id: int,
                           include_current_interests: bool = False):
    # Sum up the similarities.
    suggestions: Dict[str, float] = defaultdict(float)
    for other_user_id, similarity in most_similar_user_to(user_id):
        for interest in users_interests[other_user_id]:
            suggestions[interest] += similarity

    
    suggestions = sorted(suggestions.items(),
                         key=lambda pair: pair[-1],  #
                         reverse=True)

    
    if include_current_interests:
        return suggestions
    else:
        return [(suggestion, weight)
                for suggestion, weight in suggestions
                if suggestion not in users_interests[user_id]]
    
user_based_suggestions(0)

[('MapReduce', np.float64(0.5669467095138409)),
 ('MongoDB', np.float64(0.5070925528371099)),
 ('Postgres', np.float64(0.5070925528371099)),
 ('NoSQL', np.float64(0.33806170189140655)),
 ('neural networks', np.float64(0.1889822365046136)),
 ('deep learning', np.float64(0.1889822365046136)),
 ('artificial intelligence', np.float64(0.1889822365046136)),
 ('databases', np.float64(0.16903085094570328)),
 ('MySQL', np.float64(0.16903085094570328)),
 ('Python', np.float64(0.1543033499620919)),
 ('R', np.float64(0.1543033499620919)),
 ('C++', np.float64(0.1543033499620919)),
 ('Haskell', np.float64(0.1543033499620919)),
 ('programming languages', np.float64(0.1543033499620919))]

In [49]:
# item based collaborative filtering

# wiersze to zainteresowania a kolumny to uzytkownicy
interest_user_matrix = [[user_interest_vector[j]
                         for user_interest_vector in users_interest_vectors] 
                        for j, _ in enumerate(unique_interests)]

interest_similarities = [[cosine_sim(user_vector_i, user_vector_j)
                          for user_vector_j in interest_user_matrix]
                         for user_vector_i in interest_user_matrix]

def most_similar_interests_to(interest_id: int):
    similarities = interest_similarities[interest_id]
    pairs = [(unique_interests[other_interest_id], similarity)
             for other_interest_id, similarity in enumerate(similarities)
             if interest_id != other_interest_id and similarity > 0]
    return sorted(pairs,
                  key=lambda pair: pair[-1],
                  reverse=True)
most_similar_interests_to(0)

[('Hadoop', np.float64(0.8164965809277259)),
 ('Java', np.float64(0.6666666666666667)),
 ('MapReduce', np.float64(0.5773502691896258)),
 ('Spark', np.float64(0.5773502691896258)),
 ('Storm', np.float64(0.5773502691896258)),
 ('Cassandra', np.float64(0.40824829046386296)),
 ('artificial intelligence', np.float64(0.40824829046386296)),
 ('deep learning', np.float64(0.40824829046386296)),
 ('neural networks', np.float64(0.40824829046386296)),
 ('HBase', np.float64(0.33333333333333337))]

In [52]:
def item_based_suggestions(user_id: int,
                           include_current_interests: bool = False):
    
    suggestions = defaultdict(float)
    user_interest_vector = users_interest_vectors[user_id]
    for interest_id, is_interested in enumerate(user_interest_vector):
        if is_interested == 1:
            similar_interests = most_similar_interests_to(interest_id)
            for interest, similarity in similar_interests:
                suggestions[interest] += similarity


    suggestions = sorted(suggestions.items(),
                         key=lambda pair: pair[-1],
                         reverse=True)

    if include_current_interests:
        return suggestions
    else:
        return [(suggestion, weight)
                for suggestion, weight in suggestions
                if suggestion not in users_interests[user_id]]
item_based_suggestions(0)

[('MapReduce', np.float64(1.861807319565799)),
 ('MongoDB', np.float64(1.3164965809277258)),
 ('Postgres', np.float64(1.3164965809277258)),
 ('NoSQL', np.float64(1.2844570503761732)),
 ('MySQL', np.float64(0.5773502691896258)),
 ('databases', np.float64(0.5773502691896258)),
 ('Haskell', np.float64(0.5773502691896258)),
 ('programming languages', np.float64(0.5773502691896258)),
 ('artificial intelligence', np.float64(0.40824829046386296)),
 ('deep learning', np.float64(0.40824829046386296)),
 ('neural networks', np.float64(0.40824829046386296)),
 ('C++', np.float64(0.40824829046386296)),
 ('Python', np.float64(0.2886751345948129)),
 ('R', np.float64(0.2886751345948129))]